In [1]:
import geopandas as gpd
import pandas as pd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

#Read the files
index_walkability = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index_walkability = index_walkability.to_crs(operation_crs)

zones_girec = gpd.read_file(f'{input_file_path}/network_agreg/GEO_GIREC-SHP/GEO_GIREC.shp')
zones_girec = zones_girec.to_crs(operation_crs)

agglo_carreau = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_CARREAU_200-SHP/AGGLO_CARREAU_200.shp')
agglo_carreau = agglo_carreau.to_crs(operation_crs)

zones_communes = gpd.read_file(f'{input_file_path}/network_agreg/CAD_COMMUNE-SHP/CAD_COMMUNE.shp')
zones_communes = zones_communes.to_crs(operation_crs)

zones_communes_GE_fusionnee = gpd.read_file(f'{input_file_path}/network_agreg/CAD_COMMUNE-SHP/CAD_COMMUNES_GE_fusionnee.shp')
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.to_crs(operation_crs)


**GIREC**

In [2]:
# Spatial join
segments_girec = gpd.sjoin(index_walkability, zones_girec, how="inner", predicate="within")

# Columns to aggregate
cols = index_walkability.columns.to_list()

def weighted_mean(df, cols, weight_col):
    return (df[cols].multiply(df[weight_col], axis=0).sum() / df[weight_col].sum())

cols_to_agg = cols[3:]  # tes colonnes d'indicateurs

# Calcul pondéré
girec_stats = (
    segments_girec
        .groupby("OBJECTID")
        .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
        .reset_index()
)

# Merge back with zones_mmt polygons
zones_girec = zones_girec.merge(girec_stats, on="OBJECTID", how="left")

#Drop nan values 
zones_girec = zones_girec.dropna(subset=["indice_marchabilite"])

In [3]:
zones_girec.head()

,OBJECTID,NOM,NO_COM_FED,NO_COMM,CODE_SECT,SECT_VILLE,NUMERO,CD_SS_SECT,SHAPE_AREA,SHAPE_LEN,...,zone_apaisee,zone_pietonne,vitesse,walk_index,walk_index_unweighted,indice_marchabilite,Classe_Commodité,Classe_Attractivité,Classe_Infrastructure,Classe_Sécurité
0,31,Roulave,6620,20,00,NaN,2000020,020,1.579804e+06,7913.415859,...,0.000000,0.0,0.702850,0.354359,0.354359,0.354359,0.518545,0.021357,0.435542,0.454074
1,32,Essertines,6620,20,00,NaN,2000040,040,7.162025e+05,4123.464182,...,0.000000,0.0,0.589226,0.395281,0.395281,0.395281,0.632152,0.096591,0.382836,0.423938
2,33,La Tuilière,6620,20,00,NaN,2000010,010,1.333054e+06,6623.155220,...,0.000000,0.0,0.657926,0.287491,0.287491,0.287491,0.394931,0.026848,0.360085,0.443243
3,1,Signal,6607,7,00,NaN,0700080,080,8.244912e+05,4110.832189,...,0.121723,0.0,0.955566,0.336686,0.336686,0.336686,0.456784,0.013561,0.371754,0.554854
4,2,Veyrier - Marais,6645,48,00,NaN,4500060,060,6.213745e+05,4446.603843,...,0.026763,0.0,0.916574,0.372524,0.372524,0.372524,0.485856,0.057988,0.419284,0.515092


**Carreau 200**

In [4]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_carreau = gpd.sjoin(index_walkability, agglo_carreau, how="inner", predicate="within")

# Aggregate by mean
carreau_stats = (
    segments_carreau
    .groupby("GRID_ID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
agglo_carreau = agglo_carreau.merge(carreau_stats, on="GRID_ID", how="left")

# Drop rows with missing values (optional)
agglo_carreau = agglo_carreau.dropna(subset=["indice_marchabilite"])

In [5]:
segments_carreau.head()

,segment_id,geometry,length,bruit,temperature,conflit_usage,canopee,lac_cours_deau,fontaines,espaces_ouverts,...,PARTEMP,H_P_REGG,D_POP_HA,D_EMP_HA,POP_TOT_GG,EMP_TOT_GG,POP_TOT_00,EMP_TOT_00,GEOM_AREA,GEOM_LEN
1,000001,"LINESTRING (2500439.51 1114635.486, 2500441.15...",6.863,0.0,1.0,0.2500,0.523,0.0,0.0,0.0,...,0.000000,NaN,23.25,0.0,95,0,93,0,40000.0,800.0
2,000002,"LINESTRING (2501878.845 1118360.375, 2501878.5...",27.086,0.0,1.0,0.2500,0.000,1.0,0.0,1.0,...,NaN,NaN,0.00,0.0,0,0,0,0,40000.0,800.0
3,000003,"LINESTRING (2501886.429 1118347.344, 2501883.9...",15.077,0.0,1.0,0.2500,0.000,1.0,0.0,1.0,...,NaN,NaN,0.00,0.0,0,0,0,0,40000.0,800.0
4,000004,"LINESTRING (2496591.651 1117884.916, 2496591.6...",4.338,0.0,1.0,0.1668,0.008,0.0,0.0,0.0,...,0.043478,NaN,297.00,13.5,1230,122,1188,54,40000.0,800.0
5,000005,"LINESTRING (2496593.499 1117892.836, 2496592.0...",4.600,0.0,1.0,0.1668,0.026,0.0,0.0,0.0,...,0.043478,NaN,297.00,13.5,1230,122,1188,54,40000.0,800.0


**[Communes](https://sitg.ge.ch/donnees/cad-commune)**

In [6]:
zones_communes.head()

,OBJECTID,COMMUNE,NO_COMM,ABREVIATIO,NO_COM_FED,LIEN_WWW,SHAPE_AREA,SHAPE_LEN,geometry
0,11,Collonge-Bellerive,16,C.Be,6616,https://www.acg.ch/?q=node/158,1.069281e+07,14131.330839,"POLYGON ((2504851.598 1121190.833, 2504848.698..."
1,12,Anières,2,An,6602,https://www.acg.ch/?q=node/158,8.771935e+06,14345.239728,"POLYGON ((2507936.104 1126688.939, 2507937.789..."
2,13,Hermance,28,He,6625,https://www.acg.ch/?q=node/158,4.905640e+06,10888.811239,"POLYGON ((2507773.299 1128812.147, 2507804.379..."
3,14,Versoix,47,Vs,6644,https://www.acg.ch/?q=node/158,1.514998e+07,22048.927561,"POLYGON ((2498932.588 1130337.395, 2498892.039..."
4,1,Collex-Bossy,15,Cx,6615,https://www.acg.ch/?q=node/158,6.886683e+06,14215.534595,"POLYGON ((2498311.69 1124124.414, 2498308.252 ..."


In [7]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_communes = gpd.sjoin(index_walkability, zones_communes, how="inner", predicate="within")

# Aggregate by mean
communes_stats = (
    segments_communes
    .groupby("OBJECTID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
zones_communes = zones_communes.merge(communes_stats, on="OBJECTID", how="left")

# Drop rows with missing values (optional)
zones_communes = zones_communes.dropna(subset=["indice_marchabilite"])

In [8]:
zones_communes.head()

,OBJECTID,COMMUNE,NO_COMM,ABREVIATIO,NO_COM_FED,LIEN_WWW,SHAPE_AREA,SHAPE_LEN,geometry,bruit,...,zone_apaisee,zone_pietonne,vitesse,walk_index,walk_index_unweighted,indice_marchabilite,Classe_Commodité,Classe_Attractivité,Classe_Infrastructure,Classe_Sécurité
0,11,Collonge-Bellerive,16,C.Be,6616,https://www.acg.ch/?q=node/158,1.069281e+07,14131.330839,"POLYGON ((2504851.598 1121190.833, 2504848.698...",0.719408,...,0.107714,0.004323,0.438056,0.366757,0.366757,0.366757,0.491315,0.130766,0.408613,0.386274
1,12,Anières,2,An,6602,https://www.acg.ch/?q=node/158,8.771935e+06,14345.239728,"POLYGON ((2507936.104 1126688.939, 2507937.789...",0.824421,...,0.141090,0.013054,0.530496,0.368976,0.368976,0.368976,0.471604,0.107921,0.417236,0.440482
2,13,Hermance,28,He,6625,https://www.acg.ch/?q=node/158,4.905640e+06,10888.811239,"POLYGON ((2507773.299 1128812.147, 2507804.379...",0.845756,...,0.112856,0.199531,0.664079,0.431749,0.431749,0.431749,0.568432,0.138903,0.407013,0.519812
3,14,Versoix,47,Vs,6644,https://www.acg.ch/?q=node/158,1.514998e+07,22048.927561,"POLYGON ((2498932.588 1130337.395, 2498892.039...",0.555192,...,0.197053,0.065166,0.554402,0.441718,0.441718,0.441718,0.551816,0.164005,0.455915,0.467797
4,1,Collex-Bossy,15,Cx,6615,https://www.acg.ch/?q=node/158,6.886683e+06,14215.534595,"POLYGON ((2498311.69 1124124.414, 2498308.252 ...",0.780356,...,0.000000,0.000000,0.525870,0.356603,0.356603,0.356603,0.498661,0.087959,0.415038,0.403238


*Communes avec Genève fusionnée (1 seul polygone au lieu de 4 différents)*

In [9]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_communes_GE_fusionnee = gpd.sjoin(index_walkability, zones_communes_GE_fusionnee, how="inner", predicate="within")

# Aggregate by mean
communes_GE_fusionnee_stats = (
    segments_communes_GE_fusionnee
    .groupby("OBJECTID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.merge(communes_GE_fusionnee_stats, on="OBJECTID", how="left")

# Drop rows with missing values (optional)
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.dropna(subset=["indice_marchabilite"])

In [10]:
#save the file 
zones_girec.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_girec.gpkg"), driver="GPKG")
zones_girec.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_girec.parquet')

agglo_carreau.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_carreau200.gpkg"), driver="GPKG")
agglo_carreau.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_carreau200.parquet')

zones_communes.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_communes.gpkg"), driver="GPKG")
zones_communes.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_communes.parquet')

zones_communes_GE_fusionnee.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_communes_GE_fusionnee.gpkg"), driver="GPKG")
zones_communes_GE_fusionnee.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_communes_GE_fusionnee.parquet')

**[FREQUENCE CRIMINALITE GENEVE 2024](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)**

In [ ]:
criminalite_communes = pd.read_csv(f'{input_file_path}/STAT/CRIMINALITE/CRIMINALITE_GE_2024.csv', sep=";", header=2)

criminalite_communes = criminalite_communes.rename(columns={"Loi sur les stupéfiants (LStup) : fréquence d'infractions 2024":"freq_LStup_infra","Code pénal (CP) : fréquence d'infractions 2024":"freq_CP_infra", "Loi sur les étrangers et l’intégration (LEI) : fréquence d'infractions 2024":"freq_LEI_infra"})

#convert string to float
colonnes = ['freq_LStup_infra', 'freq_CP_infra', 'freq_LEI_infra']

criminalite_communes[colonnes] = criminalite_communes[colonnes].apply(pd.to_numeric, errors='coerce')

criminalite_communes['freq_crim_mean'] = criminalite_communes[colonnes].mean(axis=1)

#print(criminalite_communes[colonnes].dtypes)

In [ ]:
criminalite_communes.head()

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.merge(
    criminalite_communes,
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
)

In [ ]:
zones_communes_GE_fusionnee.head()

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.drop(columns=['Code', 'Libellé'])

In [ ]:
zones_communes_GE_fusionnee.head()

**[TAUX MOTORISATION CANTON GENEVE](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)**

In [ ]:
taux_motorisation_communes = pd.read_csv(f'{input_file_path}/STAT/TAUX_MOTORISATION/taux_motorisation_2024.csv', sep=";", header=2)

In [ ]:
taux_motorisation_communes.head()

In [ ]:
taux_motorisation_communes = taux_motorisation_communes.rename(columns={"Taux de motorisation 2024": "freq_voitures"})

#convert string to float
colonnes = ["freq_voitures"]

taux_motorisation_communes[colonnes] = taux_motorisation_communes[colonnes].apply(pd.to_numeric, errors='coerce')

In [ ]:
taux_motorisation_communes.head()

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.merge(
    taux_motorisation_communes,
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
)

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.drop(columns=['Code', 'Libellé'])

In [ ]:
zones_communes_GE_fusionnee.head()

**[CHOMAGE](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)**

In [ ]:
chomage_communes = pd.read_csv(f'{input_file_path}/STAT/CHOMAGE/chomage_2025.csv', sep=";", header=2)

In [ ]:
chomage_communes.head()

In [ ]:
chomage_communes = chomage_communes.rename(columns={"Chômeurs inscrits 2025": "nb_chomage"})

#convert string to float
colonnes = ["nb_chomage"]

chomage_communes[colonnes] = chomage_communes[colonnes].apply(pd.to_numeric, errors='coerce')

In [ ]:
chomage_communes.head()

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.merge(
    chomage_communes,
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
)

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.drop(columns=['Code', 'Libellé'])

In [ ]:
zones_communes_GE_fusionnee.head()

**EXPORT**

In [ ]:
zones_communes_GE_fusionnee.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_communes_GE_fusionnee_CRIM.gpkg"), driver="GPKG")
zones_communes_GE_fusionnee.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_communes_GE_fusionnee_CRIM.parquet')

**CORRELATION**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

colonnes_corr = ['walk_index', 
                 'freq_LStup_infra', 
                 'freq_CP_infra', 
                 'freq_LEI_infra', 
                 'freq_crim_mean', 
                 'freq_voitures',
                 'nb_chomage']

zones_communes_GE_fusionnee_corr = zones_communes_GE_fusionnee[colonnes_corr].corr(method='pearson')
#print(zones_communes_corr)

sns.heatmap(zones_communes_GE_fusionnee_corr, annot=True, cmap='coolwarm')
plt.title("Pearson Correlation Heatmap")
plt.show()